# TP1 — Data Preprocessing & Train a Model (Weather Prediction)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/L2Math/blob/main/session3/tp1.ipynb)

## Objective

Explore real-world weather data, handle quality issues (missing values, outliers), engineer features, and **train a model** that predicts Paris weather 6 hours ahead.

The model you save at the end of this TP will be used in **TP2** to submit to the live competition.

### Roadmap

| Section | What you do |
|---------|-------------|
| 1. Load & Explore | Understand the dataset structure |
| 2. Visualize | See temporal patterns and what "+6h prediction" means |
| 3. Missing Values | Detect, visualize, and handle missing data |
| 4. Outliers | Detect and handle extreme values |
| 5. Feature Engineering | Create useful features for prediction |
| 6. Model Training | Baseline → Linear Regression → Ensemble methods |
| 7. Save Model | Export your best model for TP2 |

---
## 1. Setup

In [1]:
!pip install scikit-learn pandas matplotlib seaborn joblib

zsh:1: command not found: pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error


ModuleNotFoundError: No module named 'matplotlib'

---
## 2. Load & Explore

The dataset contains hourly weather data for **20 European cities** over the full year 2025. Each row is one city at one hour.

Columns: `timestamp`, `city_name`, `country_code`, `latitude`, `longitude`, `temperature`, `rain`, `wind_speed`, `wind_direction`, `humidity`, `clouds`, `snow`

### Exercise 1.1 — Load the CSV

Load `weather_paris_20cities.csv`, parse timestamps, and display shape, first rows, and dtypes.

*Hint:* `pd.read_csv(..., parse_dates=['timestamp'])`, `df.shape`, `df.head()`, `df.dtypes`

In [ ]:
df = pd.read_csv("weather_paris_20cities.csv", parse_dates=["timestamp"])

# Keep the European Barcelona only (the dataset also contains Barcelona, VE)
df = df[(df["city_name"] != "Barcelona") | (df["country_code"] == "ES")].copy()

print("Shape:", df.shape)
display(df.head())
display(df.dtypes)


You should see ~184K rows (20 cities × 8760 hours) and 12 columns.

### Exercise 1.2 — Summary statistics per city

Group by city and compute mean temperature, mean wind_speed, and mean rain. Which city is warmest? Coldest?

*Hint:* `df.groupby('city_name')[['temperature', 'wind_speed', 'rain']].mean()`

In [ ]:
city_stats = (
    df.groupby("city_name")[["temperature", "wind_speed", "rain"]]
      .mean()
      .sort_values("temperature", ascending=False)
)

display(city_stats)

print("Warmest city:", city_stats.index[0])
print("Coldest city:", city_stats.index[-1])


### Exercise 1.3 — Check missing values

Count missing values per column, and per city. Are missing values evenly distributed?

*Hint:* `df.isnull().sum()`, `df.groupby('city_name').apply(lambda g: g.isnull().sum())`

**Question:** Does Paris have any missing values?

In [ ]:
missing_by_col = df.isna().sum().sort_values(ascending=False)
missing_by_city = (
    df.groupby("city_name")
      .apply(lambda g: g.isna().sum().sum())
      .sort_values(ascending=False)
      .rename("total_missing")
)

display(missing_by_col)
display(missing_by_city)


Missing values are **not perfectly evenly distributed** across cities.  
They affect the weather variables (temperature, rain, wind_speed, wind_direction, humidity, clouds, snow), while identifier columns such as `timestamp`, `city_name`, `country_code`, `latitude`, and `longitude` are complete.

In practice this suggests we should impute **per city over time**, not globally.


---
## 3. Visualize & Understand the Objective

Before cleaning the data, let's visualize it to understand temporal patterns and what we're trying to predict.

### Exercise 2.1 — Plot Paris temperature over time

Filter to Paris and plot temperature over the full year. What patterns do you see?

*Hint:* `df[df['city_name']=='Paris'].plot(x='timestamp', y='temperature')`

In [ ]:
paris = df[df["city_name"] == "Paris"].sort_values("timestamp")

plt.figure(figsize=(14, 4))
plt.plot(paris["timestamp"], paris["temperature"])
plt.title("Paris temperature over time")
plt.xlabel("Time")
plt.ylabel("Temperature (°C)")
plt.tight_layout()
plt.show()


### Exercise 2.2 — Plot multiple cities — spatial correlation

On the same plot, show January temperature for Paris, London, Barcelona, and Munich. Do nearby cities have correlated weather?

*Hint:* Filter to January, loop over cities, `plt.plot(...)` for each.

In [ ]:
cities = ["Paris", "London", "Barcelona", "Munich"]
jan = df[
    (df["city_name"].isin(cities)) &
    (df["timestamp"] >= "2025-01-01") &
    (df["timestamp"] < "2025-02-01")
].sort_values("timestamp")

plt.figure(figsize=(14, 4))
for city in cities:
    city_df = jan[jan["city_name"] == city]
    plt.plot(city_df["timestamp"], city_df["temperature"], label=city)

plt.legend()
plt.title("January temperatures: Paris / London / Barcelona / Munich")
plt.xlabel("Time")
plt.ylabel("Temperature (°C)")
plt.tight_layout()
plt.show()


### Exercise 2.3 — Map visualization — spatial patterns

Plot cities on a map (using latitude/longitude) colored by a weather metric at a given timestamp. This reveals spatial gradients (e.g., temperature decreasing from south to north).

Write a function `plot_city_map(df, metric, timestamp)` that:
1. Filters the DataFrame to the given timestamp
2. Plots a scatter of (longitude, latitude) colored by the metric value
3. Annotates each point with city name and value

*Hint:* `plt.scatter(longitudes, latitudes, c=values, cmap='coolwarm')`, `plt.colorbar()`

In [ ]:
snapshot_ts = pd.Timestamp("2025-01-15 12:00:00+00:00")
snapshot = df[df["timestamp"] == snapshot_ts].copy()

plt.figure(figsize=(8, 6))
sc = plt.scatter(
    snapshot["longitude"],
    snapshot["latitude"],
    c=snapshot["temperature"],
    s=120
)
for _, row in snapshot.iterrows():
    plt.text(row["longitude"] + 0.2, row["latitude"] + 0.1, row["city_name"], fontsize=8)

plt.colorbar(sc, label="Temperature (°C)")
plt.title(f"Spatial temperature pattern at {snapshot_ts}")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()


### Exercise 2.4 — Visualize what "+6h prediction" means

For Paris, create a column `temperature_t6` = temperature shifted by -6 hours (the value 6 hours in the future). Scatter plot `temperature` (now) vs `temperature_t6` (+6h). How correlated are they?

*Hint:* `paris['temperature_t6'] = paris['temperature'].shift(-6)`, `plt.scatter(...)`

**Question:** If you just predicted "temperature stays the same", how wrong would you be on average?

In [ ]:
paris = paris.copy()
paris["temperature_t6"] = paris["temperature"].shift(-6)

plt.figure(figsize=(6, 6))
plt.scatter(paris["temperature"], paris["temperature_t6"], alpha=0.3)
plt.xlabel("Temperature now")
plt.ylabel("Temperature at T+6h")
plt.title("Paris: temperature now vs T+6h")
plt.tight_layout()
plt.show()

corr = paris[["temperature", "temperature_t6"]].corr().iloc[0, 1]
print("Correlation:", corr)


The relationship is strongly positive: current Paris temperature is already informative about the value 6 hours later.  
So a **naive persistence baseline** (`T+6 ≈ T now`) is meaningful and will be hard to beat, especially for stable weather periods.


---
## 4. Handle Missing Values

Real-world data is messy. The dataset has ~2% missing values in non-Paris cities. We need to handle them before training.

### Exercise 3.1 — Visualize missing patterns

Create a heatmap showing the fraction of missing values per city and per feature.

*Hint:* `df.groupby('city_name')[numeric_cols].apply(lambda g: g.isnull().mean())`, `sns.heatmap(...)`

In [ ]:
missing_frac = (
    df.groupby("city_name")[["temperature", "rain", "wind_speed", "wind_direction",
                             "humidity", "clouds", "snow"]]
      .apply(lambda g: g.isna().mean())
)

plt.figure(figsize=(10, 6))
sns.heatmap(missing_frac, annot=True, fmt=".3f", cmap="Reds")
plt.title("Fraction of missing values by city and feature")
plt.tight_layout()
plt.show()


### Exercise 3.2 — Try strategies: drop, forward fill, interpolation

Pick one city with missing values (e.g., London). For its temperature column:
1. **Drop** rows with missing values — how many rows lost?
2. **Forward fill** (`ffill`) — what assumption does this make?
3. **Linear interpolation** — `interpolate(method='linear')`

Plot all three on the same chart for a short time window (e.g., one week in January).

*Hint:* Work on a copy each time. Use `.loc[start:end]` to zoom in.

In [ ]:
london = df[df["city_name"] == "London"].sort_values("timestamp").copy()

drop_series = london["temperature"].dropna()
ffill_series = london["temperature"].ffill()
interp_series = london["temperature"].interpolate(limit_direction="both")

comparison = pd.DataFrame({
    "original": london["temperature"],
    "forward_fill": ffill_series,
    "interpolation": interp_series
}, index=london["timestamp"])

display(comparison.head(12))
print("Original missing:", london["temperature"].isna().sum())
print("After ffill missing:", ffill_series.isna().sum())
print("After interpolation missing:", interp_series.isna().sum())

plt.figure(figsize=(14, 4))
plt.plot(comparison.index[:300], comparison["original"][:300], label="original", alpha=0.6)
plt.plot(comparison.index[:300], comparison["forward_fill"][:300], label="ffill")
plt.plot(comparison.index[:300], comparison["interpolation"][:300], label="interpolation")
plt.legend()
plt.title("London temperature: missing-value strategies")
plt.tight_layout()
plt.show()


### Exercise 3.3 — Apply your chosen strategy

Choose the best strategy and apply it to the full dataset. Verify there are no more missing values.

**Question:** Which strategy did you choose and why?

*Hint:* Group by city, then interpolate within each group to avoid cross-city interpolation.

In [ ]:
df_clean = df.sort_values(["city_name", "timestamp"]).copy()

weather_cols = ["temperature", "rain", "wind_speed", "wind_direction",
                "humidity", "clouds", "snow"]

for col in weather_cols:
    df_clean[col] = (
        df_clean.groupby("city_name")[col]
                .transform(lambda s: s.interpolate(limit_direction="both").ffill().bfill())
    )

display(df_clean[weather_cols].isna().sum())


I chose **interpolation per city over time**.

Why:
- this is a **time series** problem, so local temporal continuity matters;
- interpolation is smoother than forward-fill for temperature / wind;
- we apply it **city by city**, which avoids mixing different local climates.

After interpolation + edge fill, the dataset has no missing weather values left.


---
## 5. Detect & Handle Outliers

The dataset also contains ~0.5% outliers (impossible or extreme values) in non-Paris cities.

### Exercise 4.1 — Box plots — spot outliers

Create box plots for `temperature`, `humidity`, `wind_speed`, and `rain` across all cities. Do you see extreme values?

*Hint:* `fig, axes = plt.subplots(2, 2)`, `sns.boxplot(data=df, y=col, ax=axes[i])`

In [ ]:
num_cols = ["temperature", "humidity", "wind_speed", "rain"]

plt.figure(figsize=(12, 8))
for i, col in enumerate(num_cols, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(data=df_clean, x=col)
    plt.title(col)
plt.tight_layout()
plt.show()


### Exercise 4.2 — IQR-based outlier detection

For each numeric column, compute Q1, Q3, and IQR. Flag values outside `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` as outliers. How many outliers per column?

*Hint:* `Q1 = df[col].quantile(0.25)`, `IQR = Q3 - Q1`, use boolean masks.

In [ ]:
numeric_cols = df_clean.select_dtypes(include="number").columns
outlier_counts = {}

for col in numeric_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df_clean[col] < lower) | (df_clean[col] > upper)
    outlier_counts[col] = int(mask.sum())

outlier_counts = pd.Series(outlier_counts).sort_values(ascending=False)
display(outlier_counts)


### Exercise 4.3 — Clip outliers and compare

Replace outlier values by clipping to the IQR bounds. Compare distributions before/after with histograms or box plots.

*Hint:* `df[col] = df[col].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)`

In [ ]:
df_clipped = df_clean.copy()

# Clip only columns where IQR clipping makes sense.
# Rain is highly zero-inflated, so IQR would clip everything to 0.
clip_cols = ["temperature", "humidity", "wind_speed"]

for col in clip_cols:
    q1 = df_clipped[col].quantile(0.25)
    q3 = df_clipped[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df_clipped[col] = df_clipped[col].clip(lower, upper)

plt.figure(figsize=(12, 8))
for i, col in enumerate(clip_cols, 1):
    plt.subplot(3, 2, 2*i - 1)
    sns.boxplot(x=df_clean[col])
    plt.title(f"{col} before")
    plt.subplot(3, 2, 2*i)
    sns.boxplot(x=df_clipped[col])
    plt.title(f"{col} after")
plt.tight_layout()
plt.show()


---
## 6. Feature Engineering

Raw data rarely works well for ML. We need to create features that help the model learn patterns.

### Exercise 5.1 — Drop unnecessary columns

Drop `country_code` (redundant with city_name). Discuss: `latitude` and `longitude` could encode spatial relationships, but since we'll pivot cities into separate columns, we can drop them too for now.

*Hint:* `df = df.drop(columns=['country_code', 'latitude', 'longitude'])`

In [ ]:
df_model = df_clipped.drop(columns=["country_code", "latitude", "longitude"]).copy()
display(df_model.head())


### Exercise 5.2 — Create the target and pivot cities

Our goal: predict Paris temperature, wind_speed, and rain at T+6h.

Steps:
1. For Paris, create target columns: shift `temperature`, `wind_speed`, `rain` by -6 to get T+6h values
2. Pivot the data so each row = one timestamp, with columns like `Paris_temperature`, `London_temperature`, etc.
3. Add Paris lag features: T-1, T-2, ..., T-6 for temperature, wind_speed, rain

*Hint:*
- `pivot_table(index='timestamp', columns='city_name', values=[...])` 
- Flatten MultiIndex columns: `df.columns = ['_'.join(col) for col in df.columns]`
- `df['Paris_temperature_lag1'] = df['Paris_temperature'].shift(1)`

In [ ]:
weather_features = ["temperature", "rain", "wind_speed", "wind_direction",
                    "humidity", "clouds", "snow"]

pivot = df_model.pivot_table(index="timestamp", columns="city_name", values=weather_features)
pivot.columns = [f"{feat}_{city}" for feat, city in pivot.columns]

# Paris lag features
for lag in range(1, 7):
    pivot[f"temperature_Paris_lag{lag}"] = pivot["temperature_Paris"].shift(lag)
    pivot[f"wind_speed_Paris_lag{lag}"] = pivot["wind_speed_Paris"].shift(lag)
    pivot[f"rain_Paris_lag{lag}"] = pivot["rain_Paris"].shift(lag)

# Targets: Paris at T+6h
pivot["temperature_Paris_t6"] = pivot["temperature_Paris"].shift(-6)
pivot["wind_speed_Paris_t6"] = pivot["wind_speed_Paris"].shift(-6)
pivot["rain_Paris_t6"] = pivot["rain_Paris"].shift(-6)

display(pivot.head())
print("Shape:", pivot.shape)


### Exercise 5.3 — Cyclical time features

The hour of day is cyclical: hour 23 and hour 0 are close, but numerically far apart. Encoding hour as a raw integer misleads the model.

Instead, encode time as **sin/cos** pairs:
```python
sin_hour = sin(2π × hour / 24)
cos_hour = cos(2π × hour / 24)
```

This preserves the circular structure: hour 23 and hour 0 become neighbors in the sin/cos space.

*Hint:* `df['sin_hour'] = np.sin(2 * np.pi * df.index.hour / 24)`

In [ ]:
pivot["sin_hour"] = np.sin(2 * np.pi * pivot.index.hour / 24)
pivot["cos_hour"] = np.cos(2 * np.pi * pivot.index.hour / 24)

display(pivot[["sin_hour", "cos_hour"]].head(24))

# Note:
# We create these features for exploration, but we will NOT use them in the final deployed model,
# because the competition API only gives a 24-hour window without an explicit timestamp.


---
## 7. Baseline & Model Training

Now we have clean data with engineered features. Let's train models to predict Paris weather at T+6h.

**Target:** `[temperature_t6, wind_speed_t6, rain_t6]` for Paris

We'll train **3 separate models** (one per target) and compare with a naive baseline. Training separate models lets you tune each one independently and understand per-target performance clearly.

### Exercise 6.1 — Prepare train/test split and scale features

1. Define your feature columns (all city weather + lags + time features) and target columns (Paris T+6h)
2. Drop rows with NaN (from shifting)
3. Split into train/test — use a **temporal split** (first 80% for training, last 20% for testing) since this is time series data
4. **Scale features** with `StandardScaler` — fit on train only, transform both. This helps Linear Regression converge and makes feature importances comparable.

**Why temporal split?** Random splits would leak future information into training. In a real competition, your model only sees past data.

*Hint:*
- `split_idx = int(len(df) * 0.8)`, `train = df.iloc[:split_idx]`, `test = df.iloc[split_idx:]`
- `scaler = StandardScaler()`, `X_train_scaled = scaler.fit_transform(X_train)`, `X_test_scaled = scaler.transform(X_test)`

In [ ]:
dataset = pivot.dropna().copy()

# Final deployed feature set:
# - all 20-city weather values at the last observed hour (already in pivot)
# - Paris lag features
# - exclude sin/cos hour to avoid train/inference mismatch in TP2
target_cols = ["temperature_Paris_t6", "wind_speed_Paris_t6", "rain_Paris_t6"]
feature_cols = [c for c in dataset.columns if c not in target_cols + ["sin_hour", "cos_hour"]]

X = dataset[feature_cols]
y = dataset[target_cols]

split_idx = int(len(dataset) * 0.8)  # chronological split
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)


### Exercise 6.2 — Naive baseline: last known Paris value

The simplest prediction: assume Paris weather in 6 hours = Paris weather right now. Compute MAE for each target.

*Hint:* Predictions = current Paris values, compare with actual T+6h values.

In [ ]:
baseline_pred = pd.DataFrame({
    "temperature_Paris_t6": X_test["temperature_Paris"],
    "wind_speed_Paris_t6": X_test["wind_speed_Paris"],
    "rain_Paris_t6": X_test["rain_Paris"],
}, index=X_test.index)

baseline_mae = {
    col: mean_absolute_error(y_test[col], baseline_pred[col])
    for col in target_cols
}
baseline_mae


### Exercise 6.3 — Linear Regression

Train a `LinearRegression` on your features. Compute MAE for each target. Does it beat the baseline?

*Hint:* You can train one model per target or use `LinearRegression` which natively supports multi-output.

In [ ]:
lin_models = {}
lin_pred = pd.DataFrame(index=X_test.index)

for col in target_cols:
    model = LinearRegression()
    model.fit(X_train_scaled, y_train[col])
    lin_models[col] = model
    lin_pred[col] = model.predict(X_test_scaled)

linear_mae = {
    col: mean_absolute_error(y_test[col], lin_pred[col])
    for col in target_cols
}
linear_mae


### Exercise 6.4 — Gradient Boosting — 3 separate models

`GradientBoostingRegressor` does not support multi-output natively. Train **3 separate models** — one for temperature, one for wind_speed, one for rain. Compare MAE with previous approaches.

This is what `MultiOutputRegressor` does under the hood — but doing it explicitly lets you:
- See each model's performance independently
- Tune hyperparameters per target (e.g., more trees for rain)
- Understand exactly what's happening

*Hint:*
```python
gb_temperature = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
gb_temperature.fit(X_train_scaled, y_train['target_temperature'])
```

In [ ]:
# We train 3 separate models.
# For rain, the simple current-rain feature is already very strong, so we keep a lightweight linear model.

model_temperature = GradientBoostingRegressor(random_state=42)
model_wind_speed = GradientBoostingRegressor(random_state=42)
model_rain = LinearRegression()

model_temperature.fit(X_train, y_train["temperature_Paris_t6"])
model_wind_speed.fit(X_train, y_train["wind_speed_Paris_t6"])
model_rain.fit(X_train[["rain_Paris"]], y_train["rain_Paris_t6"])

gb_pred = pd.DataFrame({
    "temperature_Paris_t6": model_temperature.predict(X_test),
    "wind_speed_Paris_t6": model_wind_speed.predict(X_test),
    "rain_Paris_t6": model_rain.predict(X_test[["rain_Paris"]]),
}, index=X_test.index)

gb_mae = {
    col: mean_absolute_error(y_test[col], gb_pred[col])
    for col in target_cols
}
gb_mae


### Exercise 6.5 — Compare all approaches

Create a comparison table and bar chart showing MAE for each approach and each target.

Also compute the **competition metric** — Negated Normalized MAE:
```
score = -mean(|pred - true| / std)
```
where `std_temperature = 7.49`, `std_wind_speed = 5.05`, `std_rain = 0.40`.

*Hint:* `pd.DataFrame(results).set_index('model')`, `df.plot(kind='bar')`

In [ ]:
comparison = pd.DataFrame([
    {"model": "Baseline", **baseline_mae},
    {"model": "LinearRegression", **linear_mae},
    {"model": "Final models", **gb_mae},
]).set_index("model")

display(comparison)

comparison.plot(kind="bar", figsize=(10, 4))
plt.ylabel("MAE")
plt.title("Model comparison on Paris T+6h targets")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


---
## 8. Save Your Best Model

Save your best model so we can load it in TP2 and submit it to the competition.

### Exercise 7.1 — Save with joblib

Save your 3 models and the scaler to files. We'll load them in TP2.

*Hint:*
```python
joblib.dump(gb_temperature, 'model_temperature.pkl')
joblib.dump(gb_wind_speed, 'model_wind_speed.pkl')
joblib.dump(gb_rain, 'model_rain.pkl')
joblib.dump(scaler, 'scaler.pkl')
```

**Important:** You need to save the scaler because the Agent must apply the exact same transformation at prediction time.

In [ ]:
joblib.dump(model_temperature, "model_temperature.pkl")
joblib.dump(model_wind_speed, "model_wind_speed.pkl")
joblib.dump(model_rain, "model_rain.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Saved: model_temperature.pkl, model_wind_speed.pkl, model_rain.pkl, scaler.pkl")


---
## 9. Summary

### Key Takeaways

1. **Real data is messy** — missing values and outliers must be handled before training
2. **Visualization first** — always look at your data before applying transformations
3. **Cyclical encoding** — use sin/cos for periodic features (hour of day), not raw integers
4. **Scaling** — StandardScaler helps Linear Regression and makes features comparable
5. **Temporal splits** — for time series, never use random train/test splits
6. **Always compare to a baseline** — if your model can't beat "predict last known value", something is wrong

### What you built

| Step | What you did |
|------|-------------|
| **Explore** | Loaded 20-city weather data, checked structure and quality |
| **Visualize** | Understood temporal patterns and T+6h prediction task |
| **Clean** | Handled ~2% missing values + ~0.5% outliers |
| **Engineer** | Created lag, cross-city, and cyclical time features |
| **Scale** | Applied StandardScaler (fit on train only) |
| **Train** | Compared baseline, linear, and ensemble models |
| **Save** | Exported model + scaler |

### Next: TP2

In TP2, you'll wrap this model into an `Agent` class, test it locally (reproducing the platform's evaluation), and submit it to the live competition. You'll have ~1 month by team to iterate and climb the leaderboard.